[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/weighted_dual/weighted_dual.ipynb)

# Weighted Dual Meshes on the Sphere

Points around a centre, joined into triangles, make a polyhedron. Each face plane has a pole with respect to the unit sphere, and the poles are the vertices of the polar dual polyhedron. Seen from the centre, every point has an image on the sphere: the polyhedron's images make a triangle mesh on the sphere, and the poles' images make its dual mesh. How far a point lies from the centre does not move its image, but it tilts the faces around it, and with them the poles: the dual mesh is shaped by the distances, which the mesh on the sphere does not show. Below, the points breathe in and out along fixed directions, and both pictures follow.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from scipy.optimize import LinearConstraint, linprog, minimize
from scipy.spatial import ConvexHull

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.surfaces.weighted_dual import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()

# The number of points, how far their directions are nudged off an even spread, the seeds of the nudges
# and of their phases, their spread in distance from the centre, and the frames of one breath and the
# duration of each.
points_count = 32
jitter = 0.03
direction_seed, phase_seed = 0, 3
amplitude = 0.06
frames_count = 48
duration_ms = 80
# The spread and seed of the random distances whose hull gives the mesh the rescue starts from, every point
# on the hull; the bounds on the weights, the share of the widest margin of its conditions the rescue keeps, and the
# frames from the start to the rescue.
mesh_spread, mesh_seed = 0.06, 57
weight_bounds = (0.5, 2.0)
margin_share = 0.3
settle_frames = 48

## 1. Faces and poles

A face's pole is the point whose inner product with each of the face's vertices is one: the edges of the face joined to the centre, summed, over the volume of its vertices, `((a ^ b) + (b ^ c) + (c ^ a)) * (a ^ b ^ c).inverse()`. That is polarity about the unit sphere: a point lies in the face's plane exactly when its inner product with the pole is one, so a vertex lies on a face exactly when it has inner product one with the face's pole. On the sphere, the pole of a face lands inside the face's image when it lies, seen from the centre, on the face's side of each of the three planes through the centre and an edge; comparing its volume with each edge against the third vertex's is the test. The convex hull itself, which triangles the points make, is one step outside the algebra, taken by scipy; the linear programs of the last section are the other.

In [ ]:
def directions(count: int) -> Vector:
    """Directions spread evenly over the sphere along a Fibonacci spiral, each nudged a little at random."""
    turns = np.arange(count) + 0.5
    polar = np.arccos(1 - 2 * turns / count)
    azimuth = np.pi * (1 + 5**0.5) * turns
    spiral = np.stack([np.cos(azimuth) * np.sin(polar), np.sin(azimuth) * np.sin(polar), np.cos(polar)], axis=-1)   # [points, 3]
    nudged = spiral + jitter * np.random.default_rng(direction_seed).normal(size=(count, 3))
    return mv.vector(nudged).normalized()                                      # [points] Vector


def weighted(directions: Vector, weights: np.ndarray) -> Vector:
    """Points along the given unit directions at the distances one over their weights, the weights on the
    last axis."""
    return directions * (1 / weights)


def poles(points: Vector, faces: np.ndarray) -> Vector:
    """The pole of each face with respect to the unit sphere: the point whose inner product with each of the
    face's vertices is one, the face's edges joined to the centre, summed, over the volume of its vertices."""
    a, b, c = (points[..., faces[:, corner]] for corner in range(3))           # [..., faces] Vector each
    return ((a ^ b) + (b ^ c) + (c ^ a)) * (a ^ b ^ c).inverse()               # [..., faces] Vector


def sides(points: Vector, faces: np.ndarray, duals: Vector) -> Scalar:
    """For each face and each of its edges, positive when the face's dual vertex, seen from the centre, lies
    on the face's side of the plane through the centre and that edge, as the face's third vertex does."""
    corners = points[..., faces]                                               # [..., faces, 3] Vector
    walls = corners ^ corners[..., [1, 2, 0]]                                  # [..., faces, 3] Bivector
    return (walls ^ duals[..., None]) * (walls ^ corners[..., [2, 0, 1]]).reverse()   # [..., faces, 3] Scalar


def adjacency(faces: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Each edge of a closed triangle mesh, and the two faces either side of it."""
    edge_sides = np.sort(np.stack([faces, np.roll(faces, -1, axis=1)], axis=-1), axis=-1).reshape(-1, 2)
    paired = np.lexsort(edge_sides.T[::-1]).reshape(-1, 2)                     # each edge twice, once per face
    return edge_sides[paired[:, 0]], paired // 3


def opposite(faces: np.ndarray, edges: np.ndarray, face: np.ndarray) -> np.ndarray:
    """The vertex of each given face across from its edge."""
    return faces[face].sum(axis=-1) - edges.sum(axis=-1)


def dual_lengths(points: Vector, faces: np.ndarray) -> Scalar:
    """The signed length of each dual edge across its edge, seen from the centre: how much further the pole
    of the face on one side lies to that side of the plane through the centre and the edge than the pole
    of the face on the other side; negative where the two poles have crossed over."""
    edges, beside = adjacency(faces)                                           # [edges, 2] each
    duals = poles(points, faces)                                               # [..., faces] Vector
    wall = points[..., edges[:, 0]] ^ points[..., edges[:, 1]]                 # [..., edges] Bivector
    toward = wall ^ points[..., opposite(faces, edges, beside[:, 1])]          # [..., edges] Trivector
    return (wall ^ (duals[..., beside[:, 1]] - duals[..., beside[:, 0]])) * toward.reverse() / toward.norm()   # [..., edges] Scalar


def folds(directions: Vector, weights: np.ndarray, faces: np.ndarray) -> Scalar:
    """For each edge of the mesh, how far it bends outward, as on a convex polyhedron, or folds inward where
    negative, for points along the directions at the distances one over their weights: the volume of the
    edge's ends, the third vertex of one face beside it and the far vertex of the other, times all four
    weights. That is a sum over the four corners, each weight times the volume of the other three
    directions, with alternating signs: linear in the weights, the weights on the last axis. Each is scaled
    by the face's own volume of directions, positive on a mesh turning counter-clockwise from outside."""
    edges, beside = adjacency(faces)                                           # [edges, 2] each
    corners = np.concatenate([edges, opposite(faces, edges, beside[:, 0])[:, None],
                              opposite(faces, edges, beside[:, 1])[:, None]], axis=-1)   # [edges, 4]
    others = corners[:, [[1, 2, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]]]          # [edges, 4, 3]
    volumes = directions[others[..., 0]] ^ directions[others[..., 1]] ^ directions[others[..., 2]]   # [edges, 4] Trivector
    face = volumes[:, 3]                                                       # [edges] Trivector
    return (volumes * (weights[..., corners] * [-1, 1, -1, 1])).sum(axis=-1) * face.reverse()   # [..., edges] Scalar


def insides(directions: Vector, weights: np.ndarray, faces: np.ndarray) -> Scalar:
    """For each face and each of its edges, the signed distance of the face's pole from the plane through the
    centre and that edge, positive on the face's side, for points along the directions at the distances one
    over their weights: all three positive when the pole lands inside the face. The pole is the sum of the
    corners' weights times the face's reciprocal frame, the three vectors with inner product one with their
    own corner's direction and zero with the other two, and the planes are fixed by the directions, so this
    is linear in the weights, the weights on the last axis."""
    a, b, c = (directions[faces[:, corner]] for corner in range(3))           # [faces] Vector each
    volume = (a ^ b ^ c).inverse()                                             # [faces] Trivector
    frame = stack([(b ^ c) * volume, (c ^ a) * volume, (a ^ b) * volume], axis=-1)   # [faces, 3] Vector
    duals = (frame * weights[..., faces]).sum(axis=-1)                         # [..., faces] Vector
    corners = directions[faces]                                                # [faces, 3] Vector
    walls = (corners ^ corners[:, [1, 2, 0]]).normalized()                     # [faces, 3] Bivector
    third = walls ^ corners[:, [2, 0, 1]]                                      # [faces, 3] Trivector
    return (walls ^ duals[..., None]) * third.reverse() / third.norm()         # [..., faces, 3] Scalar


def hull(points: Vector) -> np.ndarray:
    """The faces of the convex hull of points around the centre, counter-clockwise from outside. The one
    step outside the algebra: scipy finds the triangles from the points' coordinates, and each is turned
    so that the volume of its vertices turns as x, y, z do, with the centre behind it."""
    faces = ConvexHull(points.cast(ga.subspace("x y z")).kernel).simplices    # [faces, 3]
    a, b, c = (points[faces[:, corner]] for corner in range(3))               # [faces] Vector each
    turning = ((a ^ b ^ c) * mv.xyz.reverse()) > 0                             # [faces]
    return np.where(turning[:, None], faces, faces[:, [0, 2, 1]])

## 2. Changing the distances

Take the points on the sphere and their mesh there, the spherical Delaunay mesh, and hold that mesh: its triangles never change. Each point keeps its direction and moves in and out along it, each with its own phase, so its image on the sphere never moves either. What moves is everything the distances shape: the faces tilt, the polyhedron may stop being convex, and the poles of the faces, the dual vertices, travel across the sphere; once the distances differ the dual is no longer the circumcentric one, and need not even sit well inside its triangles. That freedom, invisible in the mesh on the sphere, is the projective degree of freedom of the dual. The left panel is space: the polyhedron, blue, its vertices coloured by their distance from the sphere, blue inside and red outside, and its polar dual, orange, with the unit sphere faint between them and the parts facing away from the viewer culled. The right panel is the half of the sphere facing the viewer, with the images of both meshes.

In [ ]:
def breath() -> Iterator[tuple[Vector, np.ndarray, Vector]]:
    """At each frame: the points, each at its own distance from the centre along a fixed direction, the
    faces of the mesh they had on the sphere, held throughout, and the poles of those faces."""
    faces = hull(directions(points_count))                                     # [faces, 3]
    phases = np.random.default_rng(phase_seed).uniform(0, 2 * np.pi, points_count)
    radii = 1 + amplitude * np.sin(np.linspace(0, 2 * np.pi, frames_count, endpoint=False)[:, None] + phases)   # [frames, points]
    for points in directions(points_count) * radii:                            # [points] Vector
        yield points, faces, poles(points, faces)                              # [faces] Vector


frames = list(breath())
well_centred = np.array([(render.scalars(sides(points, faces, duals)) > 0).all(axis=-1).sum() for points, faces, duals in frames])
print(f"well centred faces over the breath: {well_centred.min()} to {well_centred.max()} of {len(frames[0][1])}")
save_figure(render.draw_complex(*frames[0]), "weighted_dual");

In [ ]:
show_animation(render.animate(frames), "weighted_dual_breath", duration_ms)

## 3. Weights that keep every dual edge positive

Take the mesh of the hull of the points at random distances along their rays, and put every point back on the sphere: the images stay where they were, but the mesh is no longer the sphere's own, and its dual tangles. Its edges cross over: the poles of the two faces beside an edge land on the wrong sides of it, a dual edge of negative length, and those are exactly the edges where the polyhedron folds inward. Negative dual lengths are what break the operators built on a dual, such as a Hodge star with negative entries.

Along a fixed direction a point is fixed by its weight, one over its distance from the centre, and in the weights every edge's fold is linear: it is the volume of the edge's ends and the two vertices across it times their four weights, which is a sum over the four corners of each weight times the volume of the other three directions. So the weights that unfold the whole mesh, every dual edge positive, form a convex cone, and finding them is a linear program, the one other step outside the algebra. The map from weights to folds comes out of the algebra directly, as the folds of one weighting per point, each with that point's weight one and the others zero. The widest margin, the largest smallest fold over all weightings, says whether a rescue exists at all; the rescue shown is the nearest one to the tangled start, the weights whose squared changes sum least while every fold keeps a third of that margin, which shares the change among many points. The pole of a face is linear in the weights too, the corners' weights times the face's reciprocal frame, and the planes through the centre and its edges are fixed, so the pole's signed distance inside each of them is linear as well; so every dual vertex lying inside its triangle is a set of linear conditions too, and joins the folds in the same programs. Since every fold is linear, along the straight line in the weights from the start to the rescue each fold crosses zero at most once: the polyhedron unfolds steadily, edge by edge, and closes in on the rescue ever more slowly over the last quarter of the frames, while the mesh on the sphere stays exactly where it was. The vertices are coloured by their distance from the sphere, blue inside and red outside, each on a stretch of the ray it moves along.

In [ ]:
def fold_map(faces: np.ndarray) -> np.ndarray:
    """The folds of every edge as a linear map of the points' weights, one column per point: the folds with
    that point's weight one and every other zero, all points as one batch."""
    return render.scalars(folds(directions(points_count), np.eye(points_count), faces)).T   # [edges, points]


def widest(conditions: np.ndarray) -> tuple[np.ndarray, float]:
    """The weights that make the smallest condition, each scaled to unit size, as large as it can be, with
    the weights within their bounds and summing to their count, and that smallest condition: negative when
    no weights meet them all."""
    scaled = conditions / np.linalg.norm(conditions, axis=-1, keepdims=True)
    count = scaled.shape[-1]
    # The weights, then the smallest fold, which is maximised.
    largest = np.concatenate([np.zeros(count), [-1.0]])
    total = np.concatenate([np.ones(count), [0.0]])[None]
    result = linprog(largest, A_ub=np.hstack([-scaled, np.ones((len(scaled), 1))]), b_ub=np.zeros(len(scaled)),
                     A_eq=total, b_eq=[count], bounds=[weight_bounds] * count + [(None, None)])
    return result.x[:-1], result.x[-1]


def inside_map(faces: np.ndarray) -> np.ndarray:
    """Every pole's signed distances inside its face as a linear map of the points' weights, one column per
    point: the distances with that point's weight one and every other zero, all points as one batch."""
    return np.moveaxis(render.scalars(insides(directions(points_count), np.eye(points_count), faces)), 0, -1)                               # [faces, 3, points]


def nearest(conditions: np.ndarray, margin: float, start: np.ndarray) -> np.ndarray:
    """The weights nearest to the start, by the summed squares of the changes, that leave every condition,
    scaled to unit size, at least the given margin: the change shared among many points rather than a few."""
    scaled = conditions / np.linalg.norm(conditions, axis=-1, keepdims=True)
    result = minimize(lambda weights: ((weights - start) ** 2).sum(), start, jac=lambda weights: 2 * (weights - start),
                      method="SLSQP", constraints=[LinearConstraint(scaled, margin, np.inf)], bounds=[weight_bounds] * len(start))
    return result.x


def timing(frames: int) -> np.ndarray:
    """How far along the way each frame is: at a steady pace for three quarters of the frames, then closing in
    on the end over the last quarter, the way left shrinking by so many e-folds, the pace continuous where it
    starts closing in."""
    closing_start, folds = 3 / 4, 3
    rate = folds / (1 - closing_start)
    left = 1 / (1 + closing_start * rate)                                      # the way left where closing in starts
    time = np.linspace(0, 1, frames)
    return np.where(time < closing_start, time * left * rate, 1 - left * np.exp(-(time - closing_start) * rate))


radii = np.exp(mesh_spread * np.random.default_rng(mesh_seed).normal(size=points_count))   # [points]
rescued_faces = hull(directions(points_count) * radii)                       # [faces, 3]
start = np.ones(points_count)                                                  # [points]
# Every fold outward, and every pole inside its face.
conditions = np.concatenate([fold_map(rescued_faces), inside_map(rescued_faces).reshape(-1, points_count)])   # [edges + faces * 3, points]
rescue = nearest(conditions, margin_share * widest(conditions)[1], start)       # [points]
settled = [(points, rescued_faces, poles(points, rescued_faces))
           for points in weighted(directions(points_count), start + timing(settle_frames)[:, None] * (rescue - start))]

for name, (points, faces, _) in (("at the start", settled[0]), ("rescued", settled[-1])):
    print(f"{name:13s} negative dual edges {(render.scalars(dual_lengths(points, faces)) < 0).sum()} of {len(adjacency(faces)[0])}")
print(f"widest margin of the conditions {widest(conditions)[1]:.4f}")
for name, (points, faces, duals) in (("at the start", settled[0]), ("rescued", settled[-1])):
    print(f"{name:13s} well-centred faces {(render.scalars(sides(points, faces, duals)) > 0).all(axis=-1).sum()} of {len(faces)}")
show_animation(render.animate(settled), "weighted_dual_rescue", duration_ms)

In [ ]:
# checks
# On the sphere the dual edges cross over exactly where the polyhedron folds; along the straight line to
# the rescue they only ever uncross, and at the rescue none is left.
crossed = [(render.scalars(dual_lengths(points, faces)) < 0).sum() for points, faces, _ in settled]
assert np.array_equal(render.scalars(dual_lengths(*settled[0][:2])) < 0,
                      render.scalars(folds(directions(points_count), np.ones(points_count), rescued_faces)) < 0)
assert crossed[0] > 0 and crossed[-1] == 0 and (np.diff(crossed) <= 0).all()
# At the rescue every pole lies inside its face.
assert (render.scalars(sides(*settled[-1])) > 0).all()
for points, faces, duals in frames:
    # Every pole has inner product one with each of its face's vertices.
    assert np.abs(render.scalars(duals[:, None] | points[faces]) - 1).max() < 1e-10


def flip(faces: np.ndarray, edge: int) -> np.ndarray:
    """The mesh with one edge turned to join the other two corners of the faces beside it."""
    edges, beside = adjacency(faces)
    first, second = beside[edge]
    a, b = edges[edge]
    c, d = faces[first].sum() - a - b, faces[second].sum() - a - b
    turned = faces[first][(list(faces[first]).index(a) + 1) % 3] != b
    a, b = (b, a) if turned else (a, b)
    flipped = faces.copy()
    flipped[first], flipped[second] = (a, d, c), (d, b, c)
    return flipped


# A mesh that is no hull at any distances has no rescue, and the same program says so: the sphere's mesh
# with so many random edges flipped, from this seed, has a negative widest margin.
flips, flip_seed = 8, 6
rng = np.random.default_rng(flip_seed)
flipped_faces = hull(directions(points_count))                                 # [faces, 3]
for _ in range(flips):
    flipped_faces = flip(flipped_faces, rng.integers(len(adjacency(flipped_faces)[0])))
assert widest(fold_map(flipped_faces))[1] < 0